# 02 — Missing Analysis

**Amaç:** Missing oranlarını, target ilişkisini ve missing feature adaylarını değerlendirmek.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display
from kaggle_s6_e7.config import (FIGURES_DIR, TABLES_DIR, TARGET_COL, ID_COL,
    PLOT_SAMPLE_SIZE, RANDOM_STATE, ensure_report_dirs)
from kaggle_s6_e7.data import load_competition_data, infer_feature_columns, validate_schema
ensure_report_dirs()
train, test = load_competition_data()
validate_schema(train, test)
cat_cols, num_cols = infer_feature_columns(train)
plot_df = train.sample(min(PLOT_SAMPLE_SIZE, len(train)), random_state=RANDOM_STATE)
sns.set_theme(style="whitegrid")

## Train/test missing tablosu

In [2]:
from kaggle_s6_e7.eda import missing_summary, missingness_target_table
missing = missing_summary(train.drop(columns=TARGET_COL), test)
display(missing[missing.train_missing_count.gt(0) | missing.test_missing_count.gt(0)])
missing.to_csv(TABLES_DIR / "02_missing_summary.csv")
missing_cols = missing.index[missing.train_missing_count.gt(0)].tolist()

,train_missing_count,train_missing_rate,test_missing_count,test_missing_rate,missing_rate_delta
stress_level,82811,0.120001,35490,0.119999,-1.854832e-06
sleep_duration,75999,0.110129,32571,0.110129,-3.723696e-07
sleep_quality,58331,0.084527,24999,0.084527,-2.858023e-07
calorie_expenditure,52853,0.076589,22652,0.076591,2.156181e-06
water_intake,43477,0.063002,18633,0.063002,-2.130227e-07
physical_activity_level,36621,0.053067,15695,0.053068,7.866265e-07
smoking_alcohol,28582,0.041418,12249,0.041416,-1.589128e-06
gender,21373,0.030971,9160,0.030972,3.783080e-07
step_count,13916,0.020166,5964,0.020165,-6.818373e-08
bmi,13898,0.020139,5956,0.020138,-1.034153e-06


## Missingness ve target

In [3]:
tables=[]
for col in missing_cols:
    table=missingness_target_table(train,col,TARGET_COL)
    table.insert(0,"feature",col); tables.append(table.reset_index())
    print(f"\n{col}"); display(table)
missing_target=pd.concat(tables,ignore_index=True)
missing_target.to_csv(TABLES_DIR / "02_missingness_target.csv",index=False)


stress_level


health_condition,feature,at-risk,fit,unhealthy
is_missing,,,,
False,stress_level,0.858638,0.057707,0.083655
True,stress_level,0.858944,0.057468,0.083588



sleep_duration


health_condition,feature,at-risk,fit,unhealthy
is_missing,,,,
False,sleep_duration,0.858594,0.057723,0.083683
True,sleep_duration,0.859327,0.057317,0.083356



sleep_quality


...

health_condition,feature,at-risk,fit,unhealthy
is_missing,,,,
False,heart_rate,0.858535,0.057706,0.083759
True,heart_rate,0.870803,0.055279,0.073918



exercise_duration


health_condition,feature,at-risk,fit,unhealthy
is_missing,,,,
False,exercise_duration,0.858753,0.057650,0.083596
True,exercise_duration,0.850891,0.060426,0.088683



diet_type


health_condition,feature,at-risk,fit,unhealthy
is_missing,,,,
False,diet_type,0.858585,0.057747,0.083668
True,diet_type,0.867555,0.050862,0.081582


## Missing count adayı

In [4]:
from kaggle_s6_e7.features import add_missing_features
feature_cols=[c for c in train.columns if c not in [ID_COL,TARGET_COL]]
train_missing=add_missing_features(train,feature_cols)
missing_count_target=pd.crosstab(train_missing.missing_count,train_missing[TARGET_COL],normalize="index")
display(missing_count_target)
missing_count_target.to_csv(TABLES_DIR / "02_missing_count_target.csv")

health_condition,at-risk,fit,unhealthy
missing_count,,,
0,0.858033,0.057273,0.084694
1,0.858669,0.057868,0.083463
2,0.861262,0.059008,0.079730
3,0.860915,0.056378,0.082707
4,0.854632,0.063556,0.081812
5,0.885057,0.057471,0.057471
6,0.777778,0.111111,0.111111


## İncelenecek kararlar
- Numeric median + categorical `missing` güvenli V1 mi?
- Hangi kolonların missing flag'i target dağılımını anlamlı değiştiriyor?
- `missing_count` tutulmalı mı?

Sonuçları `eda_findings.md` içine **Gözlem/Aday/Kabul/Red** etiketiyle aktar.